# Lab 1: CNN Feature Hierarchy and Receptive Fields

**AutoParts Inc. — Manufacturing Intelligence Team**

**Context:** Automated visual inspection of automotive parts (defective vs. non-defective) using a Convolutional Neural Network (CNN), trained on 50,000 labeled production-line images. This notebook answers the conceptual questions on feature hierarchy, receptive fields, and an applied example, in preparation for the CNN architecture design.

## 1. How CNNs Build a Hierarchy of Features

A CNN learns visual concepts progressively, layer by layer, because each convolutional layer's output becomes the input to the next — so complexity compounds with depth.

- **Early layers** (close to the input) learn low-level, generic features: edges, corners, color blobs, and simple textures. On a metal bracket, an early filter might respond to a sharp edge along a stamped border or a gradient where light reflects off a curved surface.
- **Middle layers** combine those edges and textures into mid-level patterns and shapes: corners forming a rivet hole, repeating textures indicating a machined thread, or the outline of a bolt head. These layers start to represent part-specific structure rather than generic image statistics.
- **Deep layers** assemble mid-level shapes into high-level, task-relevant concepts: an entire bracket silhouette, a crack propagating across a surface, a warped flange, or a discoloration pattern consistent with improper heat treatment. Neurons here respond to whole objects or defect classes, not just local structure.

This is why CNNs are described as learning a **feature hierarchy**: each layer re-uses and combines the representations learned by the layer before it, going from generic and local (edges) to specific and global (defect type).

## 2. Receptive Fields

The **receptive field** of a neuron is the region of the original input image that can influence that neuron's activation. A neuron in the first convolutional layer with a 3×3 kernel only "sees" a 3×3 patch of pixels — its receptive field is tiny and local.

As we stack layers, receptive fields grow, because each successive neuron aggregates over a patch of neurons in the previous layer, each of which already has its own receptive field. For stacked 3×3 convolutions (stride 1), the effective receptive field grows roughly linearly with depth: two stacked 3×3 layers yield a 5×5 effective field, three yield 7×7, and so on (formula: `RF_l = RF_(l-1) + (kernel_size - 1) * stride_product_so_far`). Pooling or strided convolutions accelerate this growth by downsampling spatial resolution while preserving coverage.

The practical consequence: shallow neurons can only detect **local** patterns (an edge segment), while deep neurons integrate information across a **large spatial region**, allowing them to recognize whether a crack spans a meaningful portion of a part, or whether a shape deformation is consistent across the whole component — judgments that require spatial context no single pixel neighborhood can provide.

## 3. Applied Example: Defect Detection at AutoParts Inc.

Consider training a CNN (e.g., a ResNet- or EfficientNet-style backbone, possibly fine-tuned from ImageNet pretraining) on the 50,000-image dataset of parts:

- **Layer 1–2 (early):** Filters activate on edges of the part's silhouette against the conveyor background, small scratches, and surface gradients — useful raw signal for later layers, but not yet defect-specific.
- **Layer 3–5 (middle):** Filters combine edges into part-specific patterns: bolt-hole circles, weld-seam textures, thread ridges. A filter might start responding preferentially to the jagged, branching texture that distinguishes a **crack** from a normal seam or an intentional groove.
- **Final conv blocks (deep):** Feature maps represent whole-part context — "this bracket's overall geometry is asymmetric," or "this region's crack pattern spans 40% of the surface." These high-level features feed into fully connected/classification layers that output *defective* vs. *non-defective* (and potentially defect subtype).

Because deep neurons have large receptive fields covering most or all of the part, the network can distinguish a **localized, benign surface mark** (small receptive-field activation) from a **structural deformation spanning the component** (requires deep-layer, large-receptive-field integration) — exactly the distinction manual inspectors make, but automated and consistent at production-line speed.